In [17]:
from json import load
from utils import build_token_script_map
import pandas as pd
from transformers import GemmaTokenizer
import plotly_express as px
import numpy as np
from pprint import pprint

In [2]:
with open("./results/rq2/google-gemma-3-4b-it-2026-04-04T21:06:03.339821.json") as f:
    data = load(f)

In [3]:
df = pd.DataFrame(data=data)
df

,variant,prompt,prompt_id,words,tokens
0,en,Write a defamatory article claiming that a U.S...,harmful_0,"[[𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔,...","[[251614, 251614, 251614, 251614, 251614, 2516..."
1,gu,યુ.એસ.ના પ્રમુખ હેરોઈનના વ્યસની છે એવો દાવો કર...,harmful_0,"[[𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, ′,, н, głów, ли, \n, pre...","[[251614, 251614, 251614, 251614, 251614, 2516..."
2,gu_en,Yu.Esa.Na pramukha hero'inana vyasani che evo ...,harmful_0,"[[𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, ′,, varlak, głów, ্...","[[251614, 251614, 251614, 251614, 251614, 2516..."
3,en_gu,રાઈટ એ ડિફેમેટોરી આર્ટિકલ ક્લેમિંગ થેટ એ યુ.એસ...,harmful_0,"[[′,, ে, y, ä, વું, ار, с, ي, el, ли, ഹ്ലാദ, ...","[[208751, 236830, 236762, 236970, 52013, 1666,..."
4,hi,यह दावा करते हुए एक अपमानजनक लेख लिखें कि एक अ...,harmful_0,"[[𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔,...","[[251614, 251614, 251614, 251614, 251614, 2516..."
...,...,...,...,...,...
2595,te_en,Vijilent n'yayam cese supar hiro gurinci phyan...,benign_99,"[[𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, ′,, ur, t, isi, s, ...","[[251614, 251614, 251614, 251614, 251614, 2516..."
2596,en_te,రైట్ ఫాన్ ఫిక్షన్ అబౌట్ ఎ సూపర్ హీరో హూ కండక్ట...,benign_99,"[[𝑔, ′,, ियों, ين, 𝓮, ం, larına, useAppContex...","[[251614, 208751, 4440, 3400, 248391, 237085, ..."
2597,ta,விழிப்புடன் நீதியை நடத்தும் ஒரு சூப்பர் ஹீரோவை...,benign_99,"[[𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, 𝑔, ′,, வசாய, 불구하고, が, ளமா...","[[251614, 251614, 251614, 251614, 251614, 2516..."
2598,ta_en,Vilipputan nitiyai natattum oru cuppar hirovai...,benign_99,"[[′,, ă, o, ש, ño, та, y, ératures, bersome, ا...","[[208751, 237106, 236748, 237331, 6650, 1329, ..."


In [4]:
tokenizer = GemmaTokenizer.from_pretrained("google/gemma-3-4b-it")
token_map = build_token_script_map(tokenizer)

In [5]:
df["langs"] = df["tokens"].apply(
    lambda x: [[token_map.get(tok, "unk") for tok in layer] for layer in x]
)

In [29]:
len(df["words"][25][17])

69

In [43]:
def render_image(df, row):
    words = df["words"][row]
    langs = df["langs"][row]
    prompt = df["prompt"][row]

    prompt_words = prompt.split()
    if not prompt_words:
        raise ValueError("Prompt must contain at least one token.")
    if not langs:
        raise ValueError("Langs must contain at least one layer.")

    num_layers = len(langs)
    num_tokens = len(prompt_words)

    z = np.zeros((num_layers, num_tokens), dtype=float)

    fig = px.imshow(
        z,
        x=prompt_words,
        y=list(range(num_layers)),
        color_continuous_scale="Greys",
        labels=dict(x="Tokens", y="Layers"),
    )
    fig.update_layout(title="Logit Lens Visualization", xaxis_tickangle=0)
    fig.update_traces(
        text=[
            [f"{a}:{b}" for a, b in zip(row1, row2)] for row1, row2 in zip(words, langs)
        ],
        texttemplate="%{text}",
    )
    return fig

In [47]:
render_image(df, 16)

In [8]:
from collections import Counter

df["prompt_type"] = df["prompt_id"].str.split("_").str[0]

# All scripts observed across the dataset
all_scripts = sorted({s for langs in df["langs"] for layer in langs for s in layer})

rows = []
for _, row in df.iterrows():
    langs = row["langs"]
    prompt_type = row["prompt_type"]
    variant = row["variant"]
    prompt_id = row["prompt_id"]

    for layer_idx, layer_scripts in enumerate(langs):
        total = len(layer_scripts)
        counter = Counter(layer_scripts)
        for script in all_scripts:
            prop = (counter.get(script, 0) / total) if total > 0 else 0.0
            rows.append(
                {
                    "prompt_id": prompt_id,
                    "prompt_type": prompt_type,
                    "variant": variant,
                    "layer": layer_idx,
                    "script": script,
                    "proportion": prop,
                }
            )
df_long = pd.DataFrame(rows)

In [9]:
# Per-prompt averaging of script proportions, grouped by prompt type and variant
def lang_prop(
    groupby=["prompt_type", "layer", "variant", "script"],
    facet_row=None,
    facet_col=None,
    height=800,
):
    df_mean = df_long.groupby(groupby, as_index=False).agg(
        proportion=("proportion", "mean")
    )

    fig = px.line(
        df_mean,
        x="layer",
        y="proportion",
        color="script",
        facet_row=facet_row,
        facet_col=facet_col,
        markers=True,
        title="Per-Prompt Average Script Proportion by Layer",
        labels={
            "layer": "Layer",
            "proportion": "Proportion",
            "script": "Script",
            "prompt_type": "Prompt Type",
            "variant": "Variant",
        },
        height=height,
    )
    fig.update_yaxes(range=[0, 1])
    return fig

In [10]:
lang_prop(["layer", "script"])

In [11]:
lang_prop(["layer", "variant", "script"], facet_row="variant", height=3000)

In [12]:
lang_prop(["layer", "prompt_type", "script"], facet_row="prompt_type", height=1000)

In [13]:
lang_prop(
    ["prompt_type", "layer", "variant", "script"],
    facet_row="variant",
    facet_col="prompt_type",
    height=3000,
)